# 5장 1강: A/B 테스트 설계 원리와 랜덤화 — 실습문제

## 실습 목표

- A/B 테스트의 대조군, 실험군, 랜덤화 단위를 데이터에서 식별합니다.
- 핵심 지표, 보조 지표, 가드레일 지표를 실험 목적에 맞게 사전에 정의합니다.
- 그룹 배정 수와 비율을 확인하고 계획한 50:50 배정과 일치하는지 점검합니다.
- 가설 → 설계 → 실행 → 분석의 순서로 온라인 실험계획을 작성합니다.

## 실습 환경 / 데이터

- Python, NumPy, pandas, SciPy
- `cookie_cats.csv`
- `userid`: 사용자 식별자
- `version`: 게임 게이트 위치(`gate_30`, `gate_40`)
- `sum_gamerounds`: 실험 기간 동안 플레이한 게임 라운드 수
- `retention_1`, `retention_7`: 설치 후 1일·7일 재방문 여부

> 이번 강의는 **실험 설계와 랜덤화 점검**이 중심입니다. 그룹 간 효과의 통계적 검정과 최종 배포 결정은 이후 강의에서 다룹니다.

## 실습 준비

아래 셀을 실행하여 데이터를 불러오고 크기, 결측치, 컬럼을 확인하세요.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

df = pd.read_csv("cookie_cats.csv")
alpha = 0.05

print(f"데이터 크기: {df.shape[0]}행, {df.shape[1]}열")
print("전체 결측치 수:", int(df.isna().sum().sum()))
print("컬럼:", df.columns.tolist())
df.head()

데이터 크기: 90189행, 5열
전체 결측치 수: 0
컬럼: ['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


---

## 필수 1. Cookie Cats 실험 구조와 지표 정의

게임의 첫 번째 강제 대기 게이트를 30단계에서 40단계로 옮기면 사용자 유지율이 달라지는지 확인하려고 합니다.

### 수행 요구사항

1. `userid`의 중복 여부를 확인하여 사용자 한 명이 한 행으로 기록되었는지 점검하세요.
2. `version`의 고유값과 그룹별 사용자 수를 확인하세요.
3. 버전별 사용자 수, 평균 게임 라운드 수, 1일 유지율, 7일 유지율을 하나의 요약표로 만드세요.
4. 아래 질문에 문장으로 답하세요.

### 질문

- 이 실험의 랜덤화 단위는 무엇인가요?
-> userid로 구분되는 사용자, 한 사용자는 하나의 버전에만 배정되어 일관된 게임 경험을 제공하여 비교

- `gate_30`과 `gate_40` 중 대조군과 실험군은 각각 무엇으로 설정할 수 있나요?
-> 기존 게이트 위치인 gate_30을 대조군, 게이트를 40단계로 옮긴 gate_40을 실험군으로 설정

- 핵심 지표, 보조 지표, 가드레일 지표를 각각 하나씩 정하고 이유를 설명하세요.
-> 핵심 지표는 단순 1일 평가보다는 7일 이후 시점을 나타내는 retention_7로(재방문)
-> 보조 지표는 초기 반응 확인을 위해 retention_1로
-> 가드레일 지표는 활동이 감소하는 지 확인하기 위해 sum_gamerounds(평균 게임 라운드 수)로 보겠다.

- 요약표에서 차이가 보인다는 사실만으로 `gate_40`의 효과라고 결론 내릴 수 있나요?
-> 아닙니다. 표본 변동으로 생길 수 있는 차이인지? 통계적으로 검정이 필요하다

In [3]:
# 여기에 코드를 작성하세요.
duplicate_users = df["userid"].duplicated().sum()
versions = df["version"].unique()
group_counts = df["version"].value_counts().reindex(["gate_30", "gate_40"])

experiment_summary = (
    df.groupby("version")
      .agg(
          users=("userid", "size"),
          avg_gamerounds=("sum_gamerounds", "mean"),
          retention_1_rate=("retention_1", "mean"),
          retention_7_rate=("retention_7", "mean"),
      ).reindex(["gate_30", "gate_40"]))

print("userid 중복 수:", duplicate_users)
print("version 고유값:", versions)
print("\n[그룹별 사용자 수]")
print(group_counts)
print("\n[버전별 지표 요약]")
print(experiment_summary.round(4))


userid 중복 수: 0
version 고유값: <StringArray>
['gate_30', 'gate_40']
Length: 2, dtype: str

[그룹별 사용자 수]
version
gate_30    44700
gate_40    45489
Name: count, dtype: int64

[버전별 지표 요약]
         users  avg_gamerounds  retention_1_rate  retention_7_rate
version                                                           
gate_30  44700         52.4563            0.4482            0.1902
gate_40  45489         51.2988            0.4423            0.1820


---

## 필수 2. 무작위 배정 실습과 그룹 비율 점검

실제 `version` 값은 변경하지 않고, 동일한 사용자 목록에 연습용 A/B 그룹을 새로 무작위 배정한 뒤 실제 배정 비율과 비교하세요.

### 수행 요구사항

1. `np.random.default_rng(42)`를 사용하세요.
2. 각 `userid`에 `A` 또는 `B`를 50:50 확률로 배정한 `randomized_users`를 만드세요.
3. 연습용 그룹별 인원수와 비율을 출력하세요.
4. 실제 `version`별 인원수와 비율도 출력하세요.
5. 실제 실험이 50:50 배정을 계획했다고 가정하고, 기대빈도를 전체 인원의 절반으로 설정하여 카이제곱 적합도 검정을 수행하세요.
6. 아래 질문에 답하세요.

### 질문

- 시드를 고정하는 이유는 무엇인가요?
-> 무작위 과정을 재현하여 수강생과 검토자가 같은 배정 결과를 얻고 코드를 확인할 수 있게 하기 위해서

- 연습용 배정에서 한 사용자가 두 그룹에 동시에 포함되지 않았는지 어떻게 확인할 수 있나요?
-> userid별로 practice_group의 고유값 수를 계산하여 최대값이 1인지 확인한다. 

- 실제 배정의 카이제곱 검정 결과는 50:50 계획과 일치한다고 볼 수 있나요?
-> p-value가 0.05보다 작으므르, 50:50 계획과 통계적으로 일치한다고 보기 어렵다

- `retention_1`, `retention_7`, `sum_gamerounds`를 랜덤화 이전 공변량의 균형 점검에 사용하면 안 되는 이유는 무엇인가요?
-> 세 변수는 버전을 경험한 이후의 측정된 결과이므로 처치영향을 받을 수 있다.
-> 균형점검에서는 실험 전에 측정된 기기, 국가 등 외부 요인 같은 공변량이 필요하지만 현 데이터에서는 제공하지 않았다


In [4]:
# 여기에 코드를 작성하세요.
rng = np.random.default_rng(42)

randomized_users = df[["userid"]].copy()
randomized_users["practice_group"] = rng.choice(["A", "B"], size=len(randomized_users), p=[0.5, 0.5])

practice_counts = randomized_users["practice_group"].value_counts().reindex(["A", "B"])
practice_ratios = randomized_users["practice_group"].value_counts(normalize=True).reindex(["A", "B"])

actual_counts = df["version"].value_counts().reindex(["gate_30", "gate_40"])
actual_ratios = df["version"].value_counts(normalize=True).reindex(["gate_30", "gate_40"])

expected_counts = np.repeat(len(df) / 2, 2)
chi2_srm, p_srm = stats.chisquare(actual_counts.to_numpy(), f_exp=expected_counts)

max_groups_per_user = randomized_users.groupby("userid")["practice_group"].nunique().max()

print("[연습용 무작위 배정 인원수]")
print(practice_counts)
print("\n[연습용 무작위 배정 비율]")
print((practice_ratios * 100).round(3).astype(str) + "%")
print("\n[실제 버전별 인원수]")
print(actual_counts)
print("\n[실제 버전별 비율]")
print((actual_ratios * 100).round(3).astype(str) + "%")
print("\n사용자당 최대 그룹 수:", max_groups_per_user)
print(f"50:50 적합도 검정: chi2 = {chi2_srm:.4f}, p-value = {p_srm:.4f}")


[연습용 무작위 배정 인원수]
practice_group
A    44852
B    45337
Name: count, dtype: int64

[연습용 무작위 배정 비율]
practice_group
A    49.731%
B    50.269%
Name: proportion, dtype: str

[실제 버전별 인원수]
version
gate_30    44700
gate_40    45489
Name: count, dtype: int64

[실제 버전별 비율]
version
gate_30    49.563%
gate_40    50.437%
Name: proportion, dtype: str

사용자당 최대 그룹 수: 1
50:50 적합도 검정: chi2 = 6.9024, p-value = 0.0086


---

## 과제. Cookie Cats A/B 테스트 전체 계획서 작성

`gate_30`을 기존 버전, `gate_40`을 새 버전으로 설정한 A/B 테스트 계획을 작성하세요.

### 수행 요구사항

1. 아래 네 단계를 모두 포함한 계획서를 작성하세요.
   - 가설 설정
   - 실험 설계
   - 실험 실행
   - 결과 분석
2. 실험 단위, 대조군, 실험군, 핵심·보조·가드레일 지표를 명시하세요.
3. 실행 전에 확인할 데이터 품질 항목을 두 가지 이상 작성하세요.
4. SUTVA 위반 또는 실험 간 간섭 가능성을 검토하세요.
5. 버전별 관측 지표를 다시 계산하되, 아직 통계적 검정을 하지 않았다는 점을 반영해 최종 의사결정을 보류하는 6~8문장의 결론을 작성하세요.

> 과제는 필수 문제와 동일한 수준입니다. 표본 크기나 MDE를 계산할 필요는 없습니다.


귀무가설: 첫 강제 대기 게이트를 30단계에서 40단계로 옮겨도 7일 유지율은 차이가 없다.
대립가설: 게이트를 40단계로 옮기면 7일 유지율이 달라진다.
검정 방향: 양측검정 / 유의수준: α = 0.05

용어	  |    쉬운 뜻  	|      이 실험에서는
실험 단위	무엇을 기준으로 나누나?	사용자 한 명 (userid)
대조군   	안 바꾼 쪽 (비교 기준)	gate_30 (기존 게이트)
실험군	    바꾼 쪽               gate_40 (게이트 옮김)
핵심 지표	최종 판단 기준 딱 하나	retention_7 (7일 뒤 돌아온 비율)
보조 지표	참고용	              retention_1 (1일 뒤 돌아온 비율)
가드레일 지표	나빠지면 안 되는 것	sum_gamerounds (평균 게임 라운드)

객관적인 판단위해 7일 유지율 하나로 사용

배정된 버전을 실험 기간 동안 고정해 한 사용자가 같은 경험을 유지한다.
실행 전 데이터 품질 점검(요구사항 3)을 통과한 뒤 분석 진행한다.
배정 비율, 결측·이상치, 로깅 누락 여부를 실험 기간 중에도 모니터링한다.

핵심 지표인 7일 유지율의 두 비율 차이를 검정하고, 효과 크기와 신뢰구간을 확인한다.
보조 지표와 가드레일 지표는 해석 보조역할.
핵심 지표가 유의하게 개선되고 가드레일이 악화되지 않은 경우에만 배포를 결정합니다.


사용자 중복 배정 여부: userid 중복 0건인지 확인. 한 사용자가 두 버전을 모두 경험하면 비교가 성립 안됨. → 확인 결과 중복 0건, 사용자당 배정 그룹 수 최대 1개.
그룹 배정 비율(SRM): 실제 배정이 계획한 50:50과 맞는지 카이제곱 적합도 검정으로 확인. → gate_30 49.563%, gate_40 50.437%, p=0.0086으로 불균형이 의심되어 원인 조사가 필요함.
결측치와 값 범위: 전체 결측치가 0건인지, version이 gate_30/gate_40 두 값만 갖는지 확인.
이상치와 논리적 모순: 게임 라운드 수가 비정상적으로 큰 사용자(봇·테스트 계정 의심)와, 라운드가 0인데 재방문 기록이 있는 모순 건수를 확인.


SUTVA는 "한 사용자가 받은 처치가 다른 사용자의 결과에 영향을 주지 않는다"는 가정
사용자 간 정보 공유: 친구 초대나 커뮤니티를 통해 "40단계에서 막힌다"는 정보가 확산되면 대조군 사용자도 행동이 달라질 수 있다.
한 사람의 다중 계정: userid 기준으로는 다른 사용자지만 실제로는 같은 사람이면 두 버전을 모두 경험.
다른 실험·이벤트와의 중첩: 같은 기간에 다른 A/B 테스트나 게임 내 이벤트가 진행되면 유지율이 변한게 게이트 때문인지 이벤트 때문인지 구분이 어렵다.

현재 데이터에는 이를 확인할 정보(기기, 국가, 실험 기간, 동시 진행 실험 목록)가 없으므로, 실제 실험에서는 로깅 단계에서 함께 수집 필요하다.


지표	       |  gate_30	  gate_40	     차이
7일 유지율	    |  19.02%	|  18.20%   |   -0.82%p
1일 유지율   	|  44.82%	|  44.23%	|   -0.59%p
평균 게임 라운드 |	52.46	 |  51.30	 |   -1.16


실제로 계산해보면 세 지표 모두 gate_40이 낮아보이는데 p=0.0086으로 50:50으로 실제 배정을 한 것인지 확인할 필요가 있음.
아직 판단할 수 없으니 배포 결정을 미룬다.

In [ ]:
# 여기에 코드를 작성하세요.

# ---- 1) 실행 전 데이터 품질 점검 ----
print("[데이터 품질 점검]")
print("1. userid 중복 수 :", int(df["userid"].duplicated().sum()))
print("2. 전체 결측치 수 :", int(df.isna().sum().sum()))
print("3. version 고유값 :", list(df["version"].unique()))

zero_rounds = int((df["sum_gamerounds"] == 0).sum())
print(f"4. 게임 라운드 0인 사용자 수 : {zero_rounds}명 "
      f"({zero_rounds / len(df) * 100:.2f}%)")

print("5. sum_gamerounds 상위 5개 :", sorted(df["sum_gamerounds"], reverse=True)[:5])
print("   sum_gamerounds 중앙값 :", df["sum_gamerounds"].median())

contradiction = int(((df["sum_gamerounds"] == 0) &
                     (df["retention_1"] | df["retention_7"])).sum())
print(f"6. 라운드 0인데 재방문 기록이 있는 모순 건수 : {contradiction}건")

# ---- 2) 그룹 배정 비율 점검 (SRM) ----
group_counts = df["version"].value_counts().reindex(["gate_30", "gate_40"])
group_ratios = df["version"].value_counts(normalize=True).reindex(["gate_30", "gate_40"])
expected_counts = np.repeat(len(df) / 2, 2)
chi2_srm, p_srm = stats.chisquare(group_counts.to_numpy(), f_exp=expected_counts)

print("\n[그룹 배정 비율 점검]")
print(group_counts)
print((group_ratios * 100).round(3).astype(str) + "%")
print(f"50:50 적합도 검정: chi2 = {chi2_srm:.4f}, p-value = {p_srm:.4f}")
print("판정:", "배정 비율 이상 없음" if p_srm > alpha
      else "배정 비율 불균형 의심(SRM) - 원인 확인 필요")

# ---- 3) 버전별 관측 지표 재계산 ----
plan_summary = (
    df.groupby("version")
      .agg(
          users=("userid", "size"),
          retention_7_rate=("retention_7", "mean"),
          retention_1_rate=("retention_1", "mean"),
          avg_gamerounds=("sum_gamerounds", "mean"),
      ).reindex(["gate_30", "gate_40"]))

print("\n[버전별 관측 지표]")
print(plan_summary.round(4))

print("\n[gate_40 - gate_30 차이]")
for col, label in [("retention_7_rate", "핵심지표 7일 유지율"),
                   ("retention_1_rate", "보조지표 1일 유지율"),
                   ("avg_gamerounds", "가드레일 평균 라운드")]:
    base = plan_summary.loc["gate_30", col]
    new = plan_summary.loc["gate_40", col]
    diff = new - base
    rel = diff / base * 100
    if "rate" in col:
        print(f"{label} : {base * 100:.2f}% -> {new * 100:.2f}% "
              f"(차이 {diff * 100:+.2f}%p, 상대 {rel:+.2f}%)")
    else:
        print(f"{label} : {base:.2f} -> {new:.2f} (차이 {diff:+.2f}, 상대 {rel:+.2f}%)")

print("\n[결과 해석]")
print("위 수치는 통계적 검정을 거치지 않은 관측값입니다.")
print("표본 변동으로 생길 수 있는 차이인지 확인되지 않았으므로 최종 배포 결정은 보류합니다.")

[데이터 품질 점검]
1. userid 중복 수 : 0
2. 전체 결측치 수 : 0
3. version 고유값 : ['gate_30', 'gate_40']
4. 게임 라운드 0인 사용자 수 : 3994명 (4.43%)
5. sum_gamerounds 상위 5개 : [49854, 2961, 2640, 2438, 2294]
   sum_gamerounds 중앙값 : 16.0
6. 라운드 0인데 재방문 기록이 있는 모순 건수 : 111건

[그룹 배정 비율 점검]
version
gate_30    44700
gate_40    45489
Name: count, dtype: int64
version
gate_30    49.563%
gate_40    50.437%
Name: proportion, dtype: str
50:50 적합도 검정: chi2 = 6.9024, p-value = 0.0086
판정: 배정 비율 불균형 의심(SRM) - 원인 확인 필요

[버전별 관측 지표]
         users  retention_7_rate  retention_1_rate  avg_gamerounds
version                                                           
gate_30  44700            0.1902            0.4482         52.4563
gate_40  45489            0.1820            0.4423         51.2988

[gate_40 - gate_30 차이]
핵심지표 7일 유지율 : 19.02% -> 18.20% (차이 -0.82%p, 상대 -4.31%)
보조지표 1일 유지율 : 44.82% -> 44.23% (차이 -0.59%p, 상대 -1.32%)
가드레일 평균 라운드 : 52.46 -> 51.30 (차이 -1.16, 상대 -2.21%)

[결과 해석]
위 수치는 통계적 검정을 거치지 않은 관측값입니다.
표본 변동으로 생길 수 있는

---

## 실습 마무리

- 어떤 문제가 있었는가?
-> 그룹별 유지율과 평균 게임 라운드만 비교하여 사용자의 고정 배정 여부, 계획한 표본 비율, 지표 사전 지정, 
-> 외부 간섭, 로깅 등의 문제를 놓칠 수 있었다.

- 어떻게 개선했는가?
-> 실험 단위를 userid로 명시하고 대조군, 실험군과 3가지 지표를 사전에 정의했다. 
-> 실제 배정 비율, 이용자 중복 체크, 50:50 적합도, 결과변수, 사전 공변량의 차이 등을 함께 점검했다.

- 무엇을 근거로 개선되었다고 판단했는가?
-> 사용자 중복 0건, 실제 그룹비율 약 49.5%, 50.4%, 적합도 검정 p-value가 0.05미만을 확인하여 단순히 비슷해 보인다는 필수1의 판단보다는 구체적인 조사 증거를 확보했다.
-> 40gate를 베포전에 기술검정을 통해 버전 업 베포 결정을 보류했다.

단순한 그룹별 지표 비교에서 놓칠 수 있는 문제와, 실험 단위·사전 지표·배정 비율·간섭 가능성을 명시하면서 설계가 어떻게 개선되었는지 정리하세요.
